In [3]:
from IPython.display import display, Markdown

def solve(z, y, mu_z, Sigma_z, W, b, Sigma_y):
    """
    Display symbolic results for a linear Gaussian system.

    Parameters — all strings, use whatever LaTeX notation you like:
        z        : latent variable name,       e.g. r'\mathbf{x}_1'
        y        : observed variable name,     e.g. r'\mathbf{x}_2'
        mu_z     : prior mean,                 e.g. r'\mathbf{0}'
        Sigma_z  : prior covariance,           e.g. r'\boldsymbol{\Sigma}'
        W        : weight matrix,              e.g. r'\mathbf{A}'
        b        : bias vector,                e.g. r'\mathbf{0}'
        Sigma_y  : likelihood covariance,      e.g. r'\boldsymbol{\Sigma}'
    """
    zero_vals = (r'0', r'\mathbf{0}', r'\boldsymbol{0}')

    # ── Derived expressions ──────────────────────────────────────────────
    mu_y     = f'{W} {mu_z}' if b in zero_vals else f'{W} {mu_z} + {b}'
    cov_yy   = f'{Sigma_y} + {W} {Sigma_z} {W}^\\top'
    cov_zy   = f'{Sigma_z} {W}^\\top'
    cov_yz   = f'{W} {Sigma_z}'
    prec_post = f'{Sigma_z}^{{-1}} + {W}^\\top {Sigma_y}^{{-1}} {W}'
    cov_post  = rf'\left( {prec_post} \right)^{{-1}}'
    inner     = (f'{W}^\\top {Sigma_y}^{{-1}} {y} + {Sigma_z}^{{-1}} {mu_z}'
                 if b in zero_vals else
                 f'{W}^\\top {Sigma_y}^{{-1}} ({y} - {b}) + {Sigma_z}^{{-1}} {mu_z}')
    mu_post   = rf'\Sigma_{{{z}|{y}}} \left[ {inner} \right]'

    # ── Given ────────────────────────────────────────────────────────────
    display(Markdown('### Given'))
    display(Markdown(rf"""
$$\begin{{align}}
p({z}) &= \mathcal{{N}}\left({z} \mid {mu_z}, {Sigma_z}\right) \\\\
p({y} \mid {z}) &= \mathcal{{N}}\left({y} \mid {W}{z} + {b}, {Sigma_y}\right)
\end{{align}}$$
"""))

    # ── Joint ────────────────────────────────────────────────────────────
    display(Markdown(f'### Joint $p({z}, {y})$'))
    display(Markdown(rf"""
$$\begin{{pmatrix}} {z} \\\\ {y} \end{{pmatrix}}
\sim \mathcal{{N}}\left(
  \begin{{pmatrix}} {mu_z} \\\\ {mu_y} \end{{pmatrix}},
  \begin{{pmatrix}}
    {Sigma_z} & {cov_zy} \\\\
    {cov_yz}  & {cov_yy}
  \end{{pmatrix}}
\right)$$
"""))

    # ── Marginal ─────────────────────────────────────────────────────────
    display(Markdown(f'### Marginal $p({y})$'))
    display(Markdown(rf"""
$$p({y}) = \mathcal{{N}}\left({y} \mid {mu_y}, {cov_yy}\right)$$
"""))

    # ── Posterior ────────────────────────────────────────────────────────
    display(Markdown(f'### Posterior $p({z} \\mid {y})$ — Bayes rule for Gaussians'))
    display(Markdown(rf"""
$$\begin{{align}}
p({z} \mid {y}) &= \mathcal{{N}}\left({z} \mid \mu_{{{z}|{y}}}, \Sigma_{{{z}|{y}}}\right) \\\\
\Sigma_{{{z}|{y}}} &= {cov_post} \\\\
\mu_{{{z}|{y}}} &= {mu_post}
\end{{align}}$$
"""))

<>:8: SyntaxWarning: invalid escape sequence '\m'
<>:8: SyntaxWarning: invalid escape sequence '\m'
C:\Users\NTres\AppData\Local\Temp\ipykernel_21488\714999516.py:8: SyntaxWarning: invalid escape sequence '\m'
  z        : latent variable name,       e.g. r'\mathbf{x}_1'


In [10]:
from IPython.display import display, Markdown

def solve(z, y, mu_z, Sigma_z, W, b, Sigma_y):
    """
    Display symbolic results for a linear Gaussian system.

    Parameters — all strings, use whatever LaTeX notation you like:
        z        : latent variable name,       e.g. r'\mathbf{x}_1'
        y        : observed variable name,     e.g. r'\mathbf{x}_2'
        mu_z     : prior mean,                 e.g. r'\mathbf{0}'
        Sigma_z  : prior covariance,           e.g. r'\boldsymbol{\Sigma}'
        W        : weight matrix,              e.g. r'\mathbf{A}'
        b        : bias vector,                e.g. r'\mathbf{0}'
        Sigma_y  : likelihood covariance,      e.g. r'\boldsymbol{\Sigma}'
    """
    zero_vals = (r'0', r'\mathbf{0}', r'\boldsymbol{0}')

    def mul(A, x):
        """'A x', but simplify to 0 if either factor is zero."""
        if A in zero_vals or x in zero_vals:
            return r'\mathbf{0}'
        return f'{A} {x}'

    def add(a, b_):
        """'a + b_', but drop the '+ b_' term if b_ is zero."""
        if b_ in zero_vals:
            return a
        return f'{a} + {b_}'

    def paren(expr):
        """Wrap expr in \\left(...\\right) if it is a compound expression (contains + or -)."""
        if any(op in expr for op in ('+', '-')):
            return rf'\left( {expr} \right)'
        return expr

    # Derived expressions
    Sigma_z_p = paren(Sigma_z)  # parenthesised if compound
    Sigma_y_p = paren(Sigma_y)
    likelihood_mean = add(f'{W}{z}', b)
    W_muz     = mul(W, mu_z)
    mu_y      = W_muz if b in zero_vals else f'{W_muz} + {b}'
    cov_yy    = f'{Sigma_y_p} + {W} {Sigma_z_p} {W}^\\top'
    cov_zy    = f'{Sigma_z_p} {W}^\\top'
    cov_yz    = f'{W} {Sigma_z_p}'
    prec_post = f'{Sigma_z_p}^{{-1}} + {W}^\\top {Sigma_y_p}^{{-1}} {W}'
    cov_post  = rf'\left( {prec_post} \right)^{{-1}}'

    Sz_inv_muz = mul(f'{Sigma_z_p}^{{-1}}', mu_z)
    if b in zero_vals:
        inner = (f'{W}^\\top {Sigma_y}^{{-1}} {y}'
                 if Sz_inv_muz in zero_vals else
                 f'{W}^\\top {Sigma_y}^{{-1}} {y} + {Sz_inv_muz}')
    else:
        inner = (f'{W}^\\top {Sigma_y}^{{-1}} ({y} - {b})'
                 if Sz_inv_muz in zero_vals else
                 f'{W}^\\top {Sigma_y}^{{-1}} ({y} - {b}) + {Sz_inv_muz}')
    mu_post = rf'\Sigma_{{{z}|{y}}} \left[ {inner} \right]'

    display(Markdown('### Given'))
    display(Markdown(rf"""
$$\begin{{align}}
p({z}) &= \mathcal{{N}}\left({z} \mid {mu_z}, {Sigma_z}\right) \\\\
p({y} \mid {z}) &= \mathcal{{N}}\left({y} \mid {W}{z} + {b}, {Sigma_y}\right)
\end{{align}}$$
"""))

    display(Markdown(f'### Joint $p({z}, {y})$'))
    display(Markdown(rf"""
$$\begin{{pmatrix}} {z} \\\\ {y} \end{{pmatrix}}
\sim \mathcal{{N}}\left(
  \begin{{pmatrix}} {mu_z} \\\\ {mu_y} \end{{pmatrix}},
  \begin{{pmatrix}}
    {Sigma_z} & {cov_zy} \\\\
    {cov_yz}  & {cov_yy}
  \end{{pmatrix}}
\right)$$
"""))

    display(Markdown(f'### Marginal $p({y})$'))
    display(Markdown(rf"""
$$p({y}) = \mathcal{{N}}\left({y} \mid {mu_y}, {cov_yy}\right)$$
"""))

    display(Markdown(f'### Posterior $p({z} \\mid {y})$ — Bayes rule for Gaussians'))
    display(Markdown(rf"""
$$\begin{{align}}
p({z} \mid {y}) &= \mathcal{{N}}\left({z} \mid \mu_{{{z}|{y}}}, \Sigma_{{{z}|{y}}}\right) \\\\
\Sigma_{{{z}|{y}}} &= {cov_post} \\\\
\mu_{{{z}|{y}}} &= {mu_post}
\end{{align}}$$
"""))


<>:8: SyntaxWarning: invalid escape sequence '\m'
<>:8: SyntaxWarning: invalid escape sequence '\m'
C:\Users\NTres\AppData\Local\Temp\ipykernel_21488\3848766484.py:8: SyntaxWarning: invalid escape sequence '\m'
  z        : latent variable name,       e.g. r'\mathbf{x}_1'


In [11]:
# ── Example: Question 1.1 ────────────────────────────────────────────────
# Markov chain:  x1 ~ N(0, Σ),  x2|x1 ~ N(A x1, Σ)
# Find p(x1 | x2)

solve(
    z       = r'\mathbf{x}_1',
    y       = r'\mathbf{x}_2',
    mu_z    = r'\mathbf{0}',
    Sigma_z = r'\boldsymbol{\Sigma}',
    W       = r'\mathbf{A}',
    b       = r'\mathbf{0}',
    Sigma_y = r'\boldsymbol{\Sigma}',
)

### Given


$$\begin{align}
p(\mathbf{x}_1) &= \mathcal{N}\left(\mathbf{x}_1 \mid \mathbf{0}, \boldsymbol{\Sigma}\right) \\\\
p(\mathbf{x}_2 \mid \mathbf{x}_1) &= \mathcal{N}\left(\mathbf{x}_2 \mid \mathbf{A}\mathbf{x}_1 + \mathbf{0}, \boldsymbol{\Sigma}\right)
\end{align}$$


### Joint $p(\mathbf{x}_1, \mathbf{x}_2)$


$$\begin{pmatrix} \mathbf{x}_1 \\\\ \mathbf{x}_2 \end{pmatrix}
\sim \mathcal{N}\left(
  \begin{pmatrix} \mathbf{0} \\\\ \mathbf{0} \end{pmatrix},
  \begin{pmatrix}
    \boldsymbol{\Sigma} & \boldsymbol{\Sigma} \mathbf{A}^\top \\\\
    \mathbf{A} \boldsymbol{\Sigma}  & \boldsymbol{\Sigma} + \mathbf{A} \boldsymbol{\Sigma} \mathbf{A}^\top
  \end{pmatrix}
\right)$$


### Marginal $p(\mathbf{x}_2)$


$$p(\mathbf{x}_2) = \mathcal{N}\left(\mathbf{x}_2 \mid \mathbf{0}, \boldsymbol{\Sigma} + \mathbf{A} \boldsymbol{\Sigma} \mathbf{A}^\top\right)$$


### Posterior $p(\mathbf{x}_1 \mid \mathbf{x}_2)$ — Bayes rule for Gaussians


$$\begin{align}
p(\mathbf{x}_1 \mid \mathbf{x}_2) &= \mathcal{N}\left(\mathbf{x}_1 \mid \mu_{\mathbf{x}_1|\mathbf{x}_2}, \Sigma_{\mathbf{x}_1|\mathbf{x}_2}\right) \\\\
\Sigma_{\mathbf{x}_1|\mathbf{x}_2} &= \left( \boldsymbol{\Sigma}^{-1} + \mathbf{A}^\top \boldsymbol{\Sigma}^{-1} \mathbf{A} \right)^{-1} \\\\
\mu_{\mathbf{x}_1|\mathbf{x}_2} &= \Sigma_{\mathbf{x}_1|\mathbf{x}_2} \left[ \mathbf{A}^\top \boldsymbol{\Sigma}^{-1} \mathbf{x}_2 \right]
\end{align}$$


In [12]:
# ── Example: Question 1.1 ────────────────────────────────────────────────
# Markov chain:  x1 ~ N(0, Σ),  x2|x1 ~ N(A x1, Σ)
# Find p(x1 | x2)

solve(
    z       = r'\mathbf{x}_2',
    y       = r'\mathbf{x}_3',
    mu_z    = r'\mathbf{0}',
    Sigma_z = r'\boldsymbol{\Sigma}+\mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^T',
    W       = r'\mathbf{A}',
    b       = r'\mathbf{0}',
    Sigma_y = r'\boldsymbol{\Sigma}',
)

### Given


$$\begin{align}
p(\mathbf{x}_2) &= \mathcal{N}\left(\mathbf{x}_2 \mid \mathbf{0}, \boldsymbol{\Sigma}+\mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^T\right) \\\\
p(\mathbf{x}_3 \mid \mathbf{x}_2) &= \mathcal{N}\left(\mathbf{x}_3 \mid \mathbf{A}\mathbf{x}_2 + \mathbf{0}, \boldsymbol{\Sigma}\right)
\end{align}$$


### Joint $p(\mathbf{x}_2, \mathbf{x}_3)$


$$\begin{pmatrix} \mathbf{x}_2 \\\\ \mathbf{x}_3 \end{pmatrix}
\sim \mathcal{N}\left(
  \begin{pmatrix} \mathbf{0} \\\\ \mathbf{0} \end{pmatrix},
  \begin{pmatrix}
    \boldsymbol{\Sigma}+\mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^T & \left( \boldsymbol{\Sigma}+\mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^T \right) \mathbf{A}^\top \\\\
    \mathbf{A} \left( \boldsymbol{\Sigma}+\mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^T \right)  & \boldsymbol{\Sigma} + \mathbf{A} \left( \boldsymbol{\Sigma}+\mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^T \right) \mathbf{A}^\top
  \end{pmatrix}
\right)$$


### Marginal $p(\mathbf{x}_3)$


$$p(\mathbf{x}_3) = \mathcal{N}\left(\mathbf{x}_3 \mid \mathbf{0}, \boldsymbol{\Sigma} + \mathbf{A} \left( \boldsymbol{\Sigma}+\mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^T \right) \mathbf{A}^\top\right)$$


### Posterior $p(\mathbf{x}_2 \mid \mathbf{x}_3)$ — Bayes rule for Gaussians


$$\begin{align}
p(\mathbf{x}_2 \mid \mathbf{x}_3) &= \mathcal{N}\left(\mathbf{x}_2 \mid \mu_{\mathbf{x}_2|\mathbf{x}_3}, \Sigma_{\mathbf{x}_2|\mathbf{x}_3}\right) \\\\
\Sigma_{\mathbf{x}_2|\mathbf{x}_3} &= \left( \left( \boldsymbol{\Sigma}+\mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^T \right)^{-1} + \mathbf{A}^\top \boldsymbol{\Sigma}^{-1} \mathbf{A} \right)^{-1} \\\\
\mu_{\mathbf{x}_2|\mathbf{x}_3} &= \Sigma_{\mathbf{x}_2|\mathbf{x}_3} \left[ \mathbf{A}^\top \boldsymbol{\Sigma}^{-1} \mathbf{x}_3 \right]
\end{align}$$


In [13]:
# ── Example: Question 1.1 ────────────────────────────────────────────────
# Markov chain:  x1 ~ N(0, Σ),  x2|x1 ~ N(A x1, Σ)
# Find p(x1 | x2)

solve(
    z       = r'\mathbf{x}_2',
    y       = r'\mathbf{x}_3',
    mu_z    = r'\mathbf{A}\mathbf{x}_1',
    Sigma_z = r'\boldsymbol{\Sigma}',
    W       = r'\mathbf{A}',
    b       = r'\mathbf{0}',
    Sigma_y = r'\boldsymbol{\Sigma}',
)

### Given


$$\begin{align}
p(\mathbf{x}_2) &= \mathcal{N}\left(\mathbf{x}_2 \mid \mathbf{A}\mathbf{x}_1, \boldsymbol{\Sigma}\right) \\\\
p(\mathbf{x}_3 \mid \mathbf{x}_2) &= \mathcal{N}\left(\mathbf{x}_3 \mid \mathbf{A}\mathbf{x}_2 + \mathbf{0}, \boldsymbol{\Sigma}\right)
\end{align}$$


### Joint $p(\mathbf{x}_2, \mathbf{x}_3)$


$$\begin{pmatrix} \mathbf{x}_2 \\\\ \mathbf{x}_3 \end{pmatrix}
\sim \mathcal{N}\left(
  \begin{pmatrix} \mathbf{A}\mathbf{x}_1 \\\\ \mathbf{A} \mathbf{A}\mathbf{x}_1 \end{pmatrix},
  \begin{pmatrix}
    \boldsymbol{\Sigma} & \boldsymbol{\Sigma} \mathbf{A}^\top \\\\
    \mathbf{A} \boldsymbol{\Sigma}  & \boldsymbol{\Sigma} + \mathbf{A} \boldsymbol{\Sigma} \mathbf{A}^\top
  \end{pmatrix}
\right)$$


### Marginal $p(\mathbf{x}_3)$


$$p(\mathbf{x}_3) = \mathcal{N}\left(\mathbf{x}_3 \mid \mathbf{A} \mathbf{A}\mathbf{x}_1, \boldsymbol{\Sigma} + \mathbf{A} \boldsymbol{\Sigma} \mathbf{A}^\top\right)$$


### Posterior $p(\mathbf{x}_2 \mid \mathbf{x}_3)$ — Bayes rule for Gaussians


$$\begin{align}
p(\mathbf{x}_2 \mid \mathbf{x}_3) &= \mathcal{N}\left(\mathbf{x}_2 \mid \mu_{\mathbf{x}_2|\mathbf{x}_3}, \Sigma_{\mathbf{x}_2|\mathbf{x}_3}\right) \\\\
\Sigma_{\mathbf{x}_2|\mathbf{x}_3} &= \left( \boldsymbol{\Sigma}^{-1} + \mathbf{A}^\top \boldsymbol{\Sigma}^{-1} \mathbf{A} \right)^{-1} \\\\
\mu_{\mathbf{x}_2|\mathbf{x}_3} &= \Sigma_{\mathbf{x}_2|\mathbf{x}_3} \left[ \mathbf{A}^\top \boldsymbol{\Sigma}^{-1} \mathbf{x}_3 + \boldsymbol{\Sigma}^{-1} \mathbf{A}\mathbf{x}_1 \right]
\end{align}$$
